# 7. Topic modeling: BERTopic (Section 4.4, Tables 4-7)

Embeddings `paraphrase-multilingual-MiniLM-L12-v2`, UMAP (n_neighbors=15, n_components=5,
cosine, random_state=42) and HDBSCAN (min_cluster_size=30). The number of topics is decided by
HDBSCAN. IRBO uses the truncated RBO of the original BERTopic notebooks. A GPU is recommended.

In [ ]:
from utils import *
import hdbscan
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from gensim import corpora
from gensim.models import CoherenceModel

In [ ]:
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
summary = {}
for platform in ["whatsapp", "telegram"]:
    docs = load_dataset(platform)["text_processed"].tolist()
    model = BERTopic(embedding_model=embedding_model,
                     umap_model=UMAP(n_neighbors=15, n_components=5, metric="cosine", random_state=SEED),
                     hdbscan_model=hdbscan.HDBSCAN(min_cluster_size=30, metric="euclidean", prediction_data=True),
                     language="portuguese")
    model.fit_transform(docs, embedding_model.encode(docs, show_progress_bar=True))

    topics = [[w for w, _ in model.get_topic(t) if w.strip()] for t in model.get_topics() if t != -1]
    texts = [d.split() for d in docs]
    dictionary = corpora.Dictionary(texts)
    coherence = {m: CoherenceModel(topics=topics, texts=texts, dictionary=dictionary,
                                   coherence=m).get_coherence() for m in ["c_v", "c_npmi"]}
    summary[NAMES[platform]] = {"k": len(topics), **coherence,
                                "irbo": irbo_truncated(topics), "diversity": topic_diversity(topics)}

    pd.DataFrame({"topic": range(len(topics)), "words": ["; ".join(t) for t in topics]}) \
      .to_csv(results_folder("07_topics_bertopic") / f"bertopic_{NAMES[platform]}_topics.csv", index=False)

summary = pd.DataFrame(summary).round(3)
summary

## Comparison with the paper (Tables 4 and 5)

In [ ]:
compare_with_paper([
    ("FW.Br k", 5, summary.loc["k", "FW.Br"]), ("FW.Br C_V", 0.659, summary.loc["c_v", "FW.Br"]),
    ("FW.Br C_NPMI", 0.231, summary.loc["c_npmi", "FW.Br"]), ("FW.Br IRBO", 0.98, summary.loc["irbo", "FW.Br"]),
    ("FW.Br diversity", 0.84, summary.loc["diversity", "FW.Br"]),
    ("FT.Br k", 26, summary.loc["k", "FT.Br"]), ("FT.Br C_V", 0.641, summary.loc["c_v", "FT.Br"]),
    ("FT.Br C_NPMI", 0.155, summary.loc["c_npmi", "FT.Br"]), ("FT.Br IRBO", 0.99, summary.loc["irbo", "FT.Br"]),
    ("FT.Br diversity", 0.78, summary.loc["diversity", "FT.Br"]),
], tolerance=0.01)